In [5]:
# load alice wonderful text data

with open("shakespeare.txt","r",encoding="utf-8") as f:
  text=f.read()
text = text[:500000]   # Use the first 500k characters
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])

In [8]:
# total unique words in the dataset
len(tokenizer.word_index)


8243

In [9]:
#split data into inp and output per sequence
input_seq=[]
max_len=0
for sentence in text.split("\n"):
  tokenize_sentence=tokenizer.texts_to_sequences([sentence])[0]
  max_len=max(max_len,len(tokenize_sentence))

  for i in range(1,len(tokenize_sentence)):
    input_seq.append(tokenize_sentence[:i+1])


In [10]:
# Add extra zeros to fulfil size according to max size sequence
padded_inp=pad_sequences(input_seq,maxlen=max_len,padding="pre")
max_len

16

In [11]:
X=padded_inp[:,:-1]
y=padded_inp[:,-1]


In [12]:
# make catagories from y(ouput)
from tensorflow.keras.utils import to_categorical
y=to_categorical(y,num_classes=8244)
y.shape

(76514, 8244)

In [13]:
from tensorflow.keras.layers import Dense,Embedding,LSTM,GRU
from tensorflow.keras.models import Sequential

In [15]:
#build archtecture of model
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(16,)),
    Embedding(input_dim=8244, output_dim=100),
    GRU(128),
    Dense(8244, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 16, 100)        │       824,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        88,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8244)           │     1,063,476 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,976,196 (7.54 MB)

 Trainable params: 1,976,196 (7.54 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# compile model
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [17]:
# train model
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    min_delta=0.001,
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(X,y,epochs=70,callbacks=[early_stop])

Epoch 1/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - accuracy: 0.0569 - loss: 6.7143
Epoch 2/70
  22/2392 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.0786 - loss: 6.1510

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


2392/2392 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.0959 - loss: 5.9754
Epoch 3/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.1136 - loss: 5.5480
Epoch 4/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.1304 - loss: 5.1574
Epoch 5/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.1524 - loss: 4.7804
Epoch 6/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.1806 - loss: 4.4226
Epoch 7/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.2206 - loss: 4.0868
Epoch 8/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.2632 - loss: 3.7732
Epoch 9/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.3060 - loss: 3.4912
Epoch 10/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.3495 - loss: 3.2391
Epoch 11/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.3879 - loss: 3.0153
Epoch 12/70
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.4226 - loss: 2.8191
Epoch 13/70
2392/2392 ━━━━━━━

In [20]:
# predict next word
text="why"
for i in range(10):
  token_text=tokenizer.texts_to_sequences([text])[0]
  padded_token_text=pad_sequences([token_text],maxlen=18,padding="pre")
  pos=np.argmax(model.predict(padded_token_text,verbose=0))
  for word,index in tokenizer.word_index.items():
    if index==pos:
      text=text+" "+word
      print(text)





why then
why then he
why then he will
why then he will say
why then he will say and
why then he will say and pray
why then he will say and pray you
why then he will say and pray you sir
why then he will say and pray you sir well
why then he will say and pray you sir well and
